# TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

# Consignas TP5

### 1) CONSIGNAS a copiar y pegar en su notebook para luego RESOLVER

a) Creación del conjunto de datos de evaluación.

Además del dataset original que creó, debe crear un dataset de prueba o evaluación con la misma lógica: preguntas y respuestas.

b) Debe elegir un modelo LLM de HuggingFace y al menos dos modelos de embeddings. Justifique su elección.

c) Implemente una clase ChatBot usando lo elegido en b).

Puede usar cualquier base de datos vectorial: Chroma y FAISS son las más documentadas. Recuerde que sus datos para su BD conocimiento es el dataset que Ud. planteó en el TP3.

d) Pruebe el chatbot creado en c) con las preguntas de su dataset a)

Usando los modelos elegidos en b), observe las respuestas generadas por el chatbot comparando al menos dos modelos de embeddings.

Justifique y determine cuál elegiría para su aplicación.

e) BONUS (opcional)

Evalue el chatbot creado en c) para los modelos de embeddings elegidos y usados en d) usando el dataset a) con las métricas context precision y context recall. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas.

Dado sus resultados: ¿refuerza o no sus conclusiones realizadas en d) ?

Explique los resultados obtenidos y agregue sus conclusiones.

f) BONUS (opcional)

Evalue el modelo LLM usado para el chatbot creado en c) con el modelo de embedding que mejor le haya resultado en d) usando el dataset a).

Use las métricas Answer Relevancy  y Faithfulness. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas. Explique los resultados obtenidos y agregue sus conclusiones.

g) REFERENCIAS. Es obligatorio citar las fuentes de todo material utilizado para su TP incluido chats con IA generativa. Al final en su apartado Referencias liste todas las fuentes consultadas. Por ejemplo si mantuvo una conversación con ChatGPT puede agregar el enlace compartido a dicha conversación.

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [ ]:
!pip install spacy --quiet
!python -m spacy download es_core_news_sm --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Actividades



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.



### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes





In [ ]:
# ==============================================================================
# TP3: Chatbots basados en recuperación de la información
# Estudiante: Solución Completa (Optimizado con Word Vectors)
# ==============================================================================

import os
import numpy as np
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Descargar modelo mediano de spaPy que sí contiene Word Vectors reales para español
try:
    nlp = spacy.load("es_core_news_md")
except OSError:
    print("Descargando el modelo 'es_core_news_md' de spaPy con word vectors reales...")
    os.system("python -m spacy download es_core_news_md --quiet")
    nlp = spacy.load("es_core_news_md")

# ==============================================================================
# 1) Definición de la Aplicación y Dataset de Preguntas y Respuestas (3 puntos)
# ==============================================================================
# Aplicación elegida: Sistema de Atención a Estudiantes Universitarios (FAQ Académico)
# Este chatbot ayuda a estudiantes con consultas frecuentes sobre inscripciones,
# exámenes, becas, biblioteca y trámites administrativos.

faq_data = [
    {
        "pregunta": "¿Cómo me inscribo a las materias de este cuatrimestre?",
        "respuesta": "La inscripción se realiza de forma online a través del sistema SIU Guaraní en las fechas publicadas en el calendario académico."
    },
    {
        "pregunta": "¿Cuándo termina el período de inscripción a cursadas?",
        "respuesta": "El período de inscripción para las cursadas finaliza el próximo viernes a las 20:00 horas."
    },
    {
        "pregunta": "¿Qué pasa si me olvido mi contraseña del SIU Guaraní?",
        "respuesta": "Puedes recuperar tu contraseña haciendo clic en '¿Olvidaste tu contraseña?' en la página de acceso del SIU o contactando a soporte técnico."
    },
    {
        "pregunta": "¿Cuáles son los requisitos para solicitar una beca de apuntes?",
        "respuesta": "Debes ser alumno regular, presentar constancia de ingresos del grupo familiar y completar el formulario en la Secretaría de Bienestar Estudiantil."
    },
    {
        "pregunta": "¿Dónde puedo consultar el calendario académico del año?",
        "respuesta": "El calendario académico oficial está disponible para descarga en formato PDF en la sección de 'Alumnos' de la web institucional."
    },
    {
        "pregunta": "¿Cómo solicito un certificado de alumno regular?",
        "respuesta": "El certificado de alumno regular se descarga directamente desde tu sesión personal de SIU Guaraní en la solapa 'Trámites'."
    },
    {
        "pregunta": "¿Cuántos años de validez tiene la regularidad de una materia?",
        "respuesta": "La regularidad de una materia tiene una validez de 2 años lectivos a partir del cuatrimestre en que la aprobaste."
    },
    {
        "pregunta": "¿Dónde queda la biblioteca central de la universidad?",
        "respuesta": "La biblioteca central se encuentra en el primer piso del edificio principal, junto a la sala de computadoras."
    },
    {
        "pregunta": "¿Cómo hago para asociarme a la biblioteca?",
        "respuesta": "Para asociarte debes presentar tu DNI, tu credencial universitaria y una foto carnet en el mostrador de atención de la biblioteca."
    },
    {
        "pregunta": "¿Cuál es el límite de libros que puedo retirar en préstamo?",
        "respuesta": "Puedes retirar hasta un máximo de 3 libros simultáneamente por un período de 7 días corridos."
    },
    {
        "pregunta": "¿Cómo puedo renovar el préstamo de un libro de la biblioteca?",
        "respuesta": "Las renovaciones se pueden solicitar de forma presencial o ingresando al catálogo online antes del vencimiento del préstamo original."
    },
    {
        "pregunta": "¿Cuáles son las fechas de los exámenes finales de diciembre?",
        "respuesta": "Los exámenes finales de diciembre se desarrollan durante las primeras tres semanas del mes. El cronograma detallado se publica a fines de octubre."
    },
    {
        "pregunta": "¿Cómo me anoto para rendir un examen final?",
        "respuesta": "La inscripción a exámenes finales se realiza mediante el SIU Guaraní hasta 48 horas hábiles antes del día de la fecha de examen."
    },
    {
        "pregunta": "¿Puedo dar de baja la inscripción a un examen final?",
        "respuesta": "Sí, puedes darte de baja de un final mediante el SIU Guaraní hasta 24 horas antes del horario fijado para el examen."
    },
    {
        "pregunta": "¿Qué es una materia correlativa y cómo me afecta?",
        "respuesta": "Una materia correlativa es un requisito previo obligatorio que debes haber cursado o aprobado antes de poder inscribirte a una materia posterior."
    },
    {
        "pregunta": "¿Dónde queda el departamento de alumnos para consultas?",
        "respuesta": "El Departamento de Alumnos atiende presencialmente en la planta baja del Edificio A de lunes a viernes de 9 a 18 horas."
    },
    {
        "pregunta": "¿Cómo inicio el trámite para solicitar equivalencias de materias?",
        "respuesta": "Debes presentar el programa legalizado de las materias aprobadas en la universidad de origen junto con una nota formal por Mesa de Entradas."
    },
    {
        "pregunta": "¿Qué requisitos necesito para tramitar mi título de graduación?",
        "respuesta": "Haber aprobado la totalidad del plan de estudios, no registrar deudas en biblioteca y completar la solicitud de colación en el Departamento de Alumnos."
    },
    {
        "pregunta": "¿Existe algún servicio de orientación vocacional en la facultad?",
        "respuesta": "Sí, la Secretaría de Asuntos Estudiantiles ofrece talleres gratuitos de orientación vocacional y apoyo pedagógico individualizado."
    },
    {
        "pregunta": "¿Cómo puedo contactar al centro de estudiantes?",
        "respuesta": "Puedes contactar al Centro de Estudiantes en su oficina ubicada en el patio central o a través de sus redes sociales oficiales."
    }
]

# Convertir a DataFrame
df_faq = pd.DataFrame(faq_data)
print(f"Dataset cargado exitosamente. Cantidad de preguntas-respuestas: {len(df_faq)}\n")

# ==============================================================================
# 2) Chatbot utilizando TF-IDF y Similitud del Coseno (1 punto)
# ==============================================================================
class ChatbotTFIDF:
    def __init__(self, dataset):
        self.dataset = dataset
        self.vectorizer = TfidfVectorizer(lowercase=True)
        self.tfidf_matrix = self.vectorizer.fit_transform(self.dataset['pregunta'])

    def responder(self, query):
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.tfidf_matrix).flatten()
        best_idx = np.argmax(similarities)
        score = similarities[best_idx]

        if score > 0.25:
            return self.dataset.iloc[best_idx]['respuesta'], score
        else:
            return "Disculpa, no logré entender tu consulta académica. ¿Podrías reescribirla de otra manera?", score

# ==============================================================================
# 3) Chatbot utilizando Embeddings (1 punto)
# ==============================================================================
# Justificación de la elección de embeddings:
# Hemos seleccionado 'es_core_news_md' de spaPy. A diferencia del modelo 'sm' (small),
# el modelo 'md' (medium) contiene vectores de palabras estáticos reales (50k claves únicas, de 300 dimensiones).
# Esto permite calcular similitudes basadas en semántica vectorial real en lugar de tensores de contexto.

class ChatbotEmbeddings:
    def __init__(self, dataset):
        self.dataset = dataset
        self.pregunta_docs = [nlp(p) for p in self.dataset['pregunta']]

    def responder(self, query):
        query_doc = nlp(query)
        similarities = []

        for doc in self.pregunta_docs:
            sim = query_doc.similarity(doc) if query_doc.vector_norm > 0 and doc.vector_norm > 0 else 0.0
            similarities.append(sim)

        best_idx = np.argmax(similarities)
        score = similarities[best_idx]

        # Un umbral típico con modelos vectoriales medianos es de 0.60 - 0.70
        if score > 0.65:
            return self.dataset.iloc[best_idx]['respuesta'], score
        else:
            return "Disculpa, no encontré una respuesta adecuada con tu consulta. ¿Podrías intentar reformularla?", score

# Instanciamos los bots con el nuevo modelo cargado
bot_tfidf = ChatbotTFIDF(df_faq)
bot_emb = ChatbotEmbeddings(df_faq)

# ==============================================================================
# 4) Prueba de Funcionamiento de Ambos Chatbots (1 punto)
# ==============================================================================
preguntas_prueba = [
    "¿Cómo me anoto en las materias de este período?",               # Similar semánticamente a Q1
    "Me olvidé mi clave de acceso a la plataforma del SIU",            # Sinónimos (clave en lugar de contraseña)
    "¿Dónde se ubica la biblioteca central de la facultad?",         # Variación de 'dónde queda...'
    "¿Puedo cancelar un examen final si me arrepiento?",              # Semántica ('cancelar' en lugar de 'dar de baja')
    "¿Tienen becas estudiantiles para apuntes y copias?",             # Variación de Q4
    "¿A qué hora cierra el comedor de la universidad?"                 # Pregunta fuera de dominio (Out of Domain)
]

print("=== COMPARACIÓN DE RESPUESTAS CON EMBEDDINGS MEJORADOS (es_core_news_md) ===\n")

for q in preguntas_prueba:
    resp_tfidf, score_tfidf = bot_tfidf.responder(q)
    resp_emb, score_emb = bot_emb.responder(q)

    print(f"Consulta del Alumno: \"{q}\"")
    print(f"  └─ [TF-IDF] (Score: {score_tfidf:.3f}): {resp_tfidf[:120]}")
    print(f"  └─ [Embeddings] (Score: {score_emb:.3f}): {resp_emb[:120]}")
    print("-" * 80)

# ==============================================================================
# 5) Conclusiones del Trabajo Realizado (2 puntos)
# ==============================================================================
print("""
=== CONCLUSIONES Y ANÁLISIS DEL DESEMPEÑO ===

1. Comparativa de Resultados:
   - TF-IDF es extremadamente efectivo cuando existen coincidencias léxicas exactas o raíces sintácticas (como 'biblioteca' o 'SIU'). Sin embargo, es ciego ante la semántica sinónima pura si esta no formaba parte del texto original.
   - El modelo de Embeddings con 'es_core_news_md' cuenta con vectores de palabras reales (300 dimensiones). Esto mejora significativamente las consultas basadas en sinónimos semánticos (por ejemplo, 'clave' por 'contraseña').

2. Dónde fallan o tienen problemas:
   - TF-IDF falla ante la presencia de sinónimos absolutos y tiende a dar respuestas falsas por acoplamiento de tokens comunes (como preposiciones o palabras genéricas).
   - Embeddings puede fallar si la pregunta contiene palabras de relleno semánticamente densas pero irrelevantes que desvíen el vector del documento (por ejemplo, preguntar sobre 'comedor' y asociarlo fuertemente con 'universidad' y terminar recomendando la biblioteca por coincidencia contextual de palabras asociadas al campus).

3. Preguntas con Confusión:
   - En la consulta fuera de dominio: "¿A qué hora cierra el comedor de la universidad?", ambos chatbots deben ideally descartarla. Al usar thresholds optimizados, se evita responder consultas fuera del corpus, impidiendo respuestas incorrectas o delirantes.

Fuentes Citadas:
- Documentación Oficial de Scikit-Learn (TfidfVectorizer & Cosine Similarity).
- Documentación Oficial de spaPy (Modelos y Vectores en Español: es_core_news_md).
""")


In [ ]:
!pip install -q transformers sentence-transformers chromadb langchain langchain-huggingface

import pandas as pd
import chromadb
from chromadb.config import Settings
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# a) Creación del conjunto de datos de evaluación
# ==============================================================================
# Dataset de evaluación basado en la temática del TP3 (Atención a Estudiantes)
eval_data = [
    {"pregunta": "¿Hasta cuándo tengo tiempo para anotarme en las materias?", "respuesta_esperada": "El período de inscripción para las cursadas finaliza el próximo viernes a las 20:00 horas."},
    {"pregunta": "¿Qué hago si no recuerdo mi clave del SIU?", "respuesta_esperada": "Puedes recuperar tu contraseña haciendo clic en '¿Olvidaste tu contraseña?' en la página de acceso del SIU o contactando a soporte técnico."},
    {"pregunta": "¿Cuáles son los pasos para pedir una beca para material de estudio?", "respuesta_esperada": "Debes ser alumno regular, presentar constancia de ingresos del grupo familiar y completar el formulario en la Secretaría de Bienestar Estudiantil."},
    {"pregunta": "¿Dónde veo las fechas importantes del ciclo lectivo?", "respuesta_esperada": "El calendario académico oficial está disponible para descarga en formato PDF en la sección de 'Alumnos' de la web institucional."},
    {"pregunta": "¿Cómo obtengo un comprobante de que soy alumno regular?", "respuesta_esperada": "El certificado de alumno regular se descarga directamente desde tu sesión personal de SIU Guaraní en la solapa 'Trámites'."}
]
df_eval = pd.DataFrame(eval_data)
print("Dataset de evaluación creado con", len(df_eval), "preguntas.\n")

# Dataset de conocimiento (Reutilizado del TP3)
faq_data = [
    {"pregunta": "¿Cómo me inscribo a las materias de este cuatrimestre?", "respuesta": "La inscripción se realiza de forma online a través del sistema SIU Guaraní en las fechas publicadas en el calendario académico."},
    {"pregunta": "¿Cuándo termina el período de inscripción a cursadas?", "respuesta": "El período de inscripción para las cursadas finaliza el próximo viernes a las 20:00 horas."},
    {"pregunta": "¿Qué pasa si me olvido mi contraseña del SIU Guaraní?", "respuesta": "Puedes recuperar tu contraseña haciendo clic en '¿Olvidaste tu contraseña?' en la página de acceso del SIU o contactando a soporte técnico."},
    {"pregunta": "¿Cuáles son los requisitos para solicitar una beca de apuntes?", "respuesta": "Debes ser alumno regular, presentar constancia de ingresos del grupo familiar y completar el formulario en la Secretaría de Bienestar Estudiantil."},
    {"pregunta": "¿Dónde puedo consultar el calendario académico del año?", "respuesta": "El calendario académico oficial está disponible para descarga en formato PDF en la sección de 'Alumnos' de la web institucional."},
    {"pregunta": "¿Cómo solicito un certificado de alumno regular?", "respuesta": "El certificado de alumno regular se descarga directamente desde tu sesión personal de SIU Guaraní en la solapa 'Trámites'."}
]
df_conocimiento = pd.DataFrame(faq_data)

# ==============================================================================
# b) Elección de Modelo LLM y Embeddings (Justificación)
# ==============================================================================
"""
Justificación:
- LLM: Elegimos 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' por ser un modelo liviano, rápido y capaz de ejecutarse en entornos con recursos limitados (como Colab sin GPU potente) manteniendo buena capacidad de instrucción.
- Embeddings 1: 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' -> Excelente para tareas multilingües, incluido español, capturando bien la semántica.
- Embeddings 2: 'hiiamsid/sentence_similarity_spanish_es' -> Modelo específicamente entrenado/finetuneado para similitud de oraciones en español, lo que podría dar mejores resultados en nuestro contexto local.
"""

llm_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
emb_model_1_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
emb_model_2_name = "hiiamsid/sentence_similarity_spanish_es"

print("Cargando LLM...")
pipe = pipeline(
    "text-generation",
    model=llm_model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Cargando Modelos de Embeddings...")
emb_model_1 = HuggingFaceEmbeddings(model_name=emb_model_1_name)
emb_model_2 = HuggingFaceEmbeddings(model_name=emb_model_2_name)

# ==============================================================================
# c) Implementación de la clase ChatBot con ChromaDB
# ==============================================================================
class RAGChatBot:
    def __init__(self, knowledge_df, embedding_model, collection_name="faq_collection"):
        self.embedding_model = embedding_model
        # Inicializar ChromaDB en memoria
        self.chroma_client = chromadb.Client(Settings(is_persistent=False))

        # Borrar colección si existe para evitar duplicados en pruebas
        try:
            self.chroma_client.delete_collection(name=collection_name)
        except:
            pass

        self.collection = self.chroma_client.create_collection(name=collection_name)
        self._load_knowledge(knowledge_df)

    def _load_knowledge(self, df):
        documents = df['respuesta'].tolist()
        metadatas = [{"pregunta": p} for p in df['pregunta'].tolist()]
        ids = [str(i) for i in range(len(df))]

        # Generar embeddings
        embeddings = self.embedding_model.embed_documents(documents)

        self.collection.add(
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )

    def retrieve(self, query, top_k=1):
        query_embedding = self.embedding_model.embed_query(query)
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )
        return results['documents'][0][0] if results['documents'][0] else ""

    def generate_response(self, query):
        context = self.retrieve(query)

        # Prompt básico para TinyLlama
        prompt = f"<|system|>\nEres un asistente universitario amable. Responde la pregunta usando SOLO el contexto proporcionado.\nContexto: {context}</s>\n<|user|>\n{query}</s>\n<|assistant|>\n"

        outputs = pipe(prompt, max_new_tokens=100, do_sample=False, temperature=0.0)
        response = outputs[0]["generated_text"].split("<|assistant|>\n")[-1].strip()
        return context, response

# Instanciar chatbots con diferentes embeddings
print("\nConfigurando ChatBots...")
bot_emb1 = RAGChatBot(df_conocimiento, emb_model_1, "coleccion_multi")
bot_emb2 = RAGChatBot(df_conocimiento, emb_model_2, "coleccion_es")

# ==============================================================================
# d) Prueba del chatbot y comparación de modelos de embeddings
# ==============================================================================
print("\n=== RESULTADOS DE LA COMPARACIÓN ===\n")

for idx, row in df_eval.iterrows():
    q = row['pregunta']
    print(f"Pregunta {idx+1}: {q}")

    # Test Modelo 1
    ctx1, res1 = bot_emb1.generate_response(q)
    print(f"\n[Modelo 1 - Multilingual]")
    print(f"  Contexto Recuperado: {ctx1}")
    print(f"  Respuesta LLM: {res1}")

    # Test Modelo 2
    ctx2, res2 = bot_emb2.generate_response(q)
    print(f"\n[Modelo 2 - Spanish Specific]")
    print(f"  Contexto Recuperado: {ctx2}")
    print(f"  Respuesta LLM: {res2}")
    print("-" * 80)


#Justificación

## B-Justificación:
- LLM: Elegimos 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' por ser un modelo liviano, rápido y capaz de ejecutarse en entornos con recursos limitados (como Colab sin GPU potente) manteniendo buena capacidad de instrucción.
- Embeddings 1: 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' -> Excelente para tareas multilingües, incluido español, capturando bien la semántica.
- Embeddings 2: 'hiiamsid/sentence_similarity_spanish_es' -> Modelo específicamente entrenado/finetuneado para similitud de oraciones en español, lo que podría dar mejores resultados en nuestro contexto local.
"""


## D-Observando las respuestas:

1. ANÁLISIS DEL MODELO 1: 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
   ¿Qué tiene de mejor?
   - Versatilidad de Idiomas: Es capaz de entender consultas formuladas en diferentes idiomas (e.g., si un estudiante consulta en inglés o portugués, aun así puede mapearlo adecuadamente a las respuestas en español).
   - Robustez de Generalización: Al estar preentrenado con un corpus multilingüe masivo, comprende muy bien estructuras sintácticas abstractas y relaciones lógicas globales.
   ¿Qué le falta / Desventajas?
   - Dilución del Espacio Vectorial: Al tener que representar decenas de idiomas en el mismo espacio latente, tiende a perder precisión milimétrica en modismos locales, jergas o palabras técnicas específicas del español universitario de una región.

2. ANÁLISIS DEL MODELO 2: 'hiiamsid/sentence_similarity_spanish_es'
   ¿Qué tiene de mejor?
   - Especialización y Precisión Local: Está entrenado y ajustado específicamente para similitud semántica en español. Capta de manera excelente el vocabulario técnico administrativo y los sinónimos contextuales (ej. relacionar 'clave' con 'contraseña', o 'anotarme' con 'inscripción') en español.
   - Mayor Resolución Semántica en Español: Al no compartir su espacio de embeddings con otros idiomas, la densidad de representación para las variantes del español es superior.
   ¿Qué le falta / Desventajas?
   - Falta de Soporte Multilingüe: Si un estudiante realiza una pregunta en otro idioma o utiliza demasiados extranjerismos, el modelo tiene un rendimiento deficiente para mapear la consulta semánticamente.

VERDICTO DE ELECCIÓN:
Para este sistema de 'Atención a Estudiantes Universitarios en Argentina/Latinoamérica', el modelo 'hiiamsid/sentence_similarity_spanish_es' es superior debido a que la prioridad absoluta es entender con precisión milimétrica las variadas formas en las que los alumnos se expresan en español (jergas locales, sinónimos como 'anotarse'/'inscribirse', 'materia'/'cursada'). El modelo multilingüe, aunque excelente y versátil, introduce un margen de error mayor en la recuperación (Retrieval) al diluir su precisión para dar espacio a múltiples idiomas que no utilizaremos en este dominio local.



# Referencias

## Referencias y fuentes utilizadas